In [5]:
import pandas as pd 
import numpy as np
import wrds

In [ ]:
db = wrds.Connection(username = 'hcdb')

In [ ]:
# Get Compustat Annual Information

funda = db.raw_sql('''
                   SELECT c.cik, f.gvkey, f.cusip, f.datadate, f.fyear, f.sic, f.naics,
                          f.sale,f.revt,f.cogs,f.xsga,f.dp,f.xrd,f.xad,f.ib,f.ebitda,f.ebit,f.nopi,f.spi,f.pi,f.txp,f.ni,f.txfed,f.txfo,f.txt,f.xint,
                          f.capx,f.oancf,f.dvt,f.ob,f.gdwlia,f.gdwlip,f.gwo,
                          f.rect,f.act,f.che,f.ppegt,f.invt,f.at,f.aco,f.intan,f.ao,f.ppent,f.gdwl,f.fatb,f.fatl,
                          f.lct,f.dlc,f.dltt,f.lt,f.dm,f.dcvt,f.cshrc,f.dcpstk,f.pstk,f.ap,f.lco,f.lo,f.drc,f.drlt,f.txdi,
                          f.ceq,f.scstkc,f.emp,f.csho,
                          f.prcc_f
                   FROM comp.company AS c 
                   JOIN comp.funda AS f
                   ON f.gvkey = c.gvkey
                   WHERE datadate >='1975-01-01'
                   AND f.indfmt='INDL'
                   AND f.datafmt='STD'
                   AND f.popsrc='D'
                   AND f.consol='C'
                   ''').reset_index(drop=True)

funda_raw = funda.copy()

# this table actually have two tables comp.company and comp.funda, and  gvkey, sic and
# naics are in both so have to state from which table
#  drop=True discards the old index instead of inserting it back as a new column named index.             
                          

In [ ]:
# check dypes of SIC, NAICS and DATE
# don't forget the head() the data

In [ ]:
# Clean Compustat Annual Information

# funda = funda_raw.copy().reset_index(drop=True)

# In-SQL code 

funda['sic2'] = funda['sic'].str.slice(0,2)
funda['cnum'] = funda['cusip'].str.strip().str.slice(0,6) # remove first the whitespaces if any then keep 0 to 6
funda['prcc_f'] = funda['prcc_f'].abs()
funda['mve_f'] = funda['prcc_f']*funda['csho']
funda = funda.dropna(subset=['at','prcc_f','ni'])
funda['datadate'] = pd.to_datetime(funda['datadate'])

# sort and clean up

funda = funda.sort_values(by=['gvkey','datadate']) # by sorting we enable it to keep the first one
funda = funda.drop_duplicates(subset = ['gvkey','datadate'])

#  prep for clean-up and using time series of variables

funda['count'] = funda.groupby('gvkey').cumcount() + 1

# do some clean up, several of these variables have lots of missing values

funda['dr'] = np.where(funda['drc'].notna() & funda['drlt'].notna(), funda['drc'] + funda['drlt'], # since x + NA = NA in python
                       np.where(funda['drc'].notna() & funda['drlt'].isnull(), funda['drc'],
                                np.where(funda['drlt'].notna() & funda['drc'].isnull(), funda['drlt'], np.nan)))  

funda['dc'] = np.where(funda['dcvt'].isnull() & funda['dcpstk'].notna() & funda['pstk'].notna() & (funda['dcpstk'] > funda['pstk']), funda['dcpstk'] - funda['pstk'],
               np.where(funda['dcvt'].isnull() & funda['dcpstk'].notna() & funda['pstk'].isnull(), funda['dcpstk'],
                        funda['dcvt']))

funda['xint0'] = funda['xint'].fillna(0)
funda['xsga0'] = funda['xsga'].fillna(0) # Prof. Green agreed of this corrected version

# adding format change (not in the code)

funda['sic'] = pd.to_numeric(funda['sic'], errors='coerce')
funda['naics'] = funda['naics'].str.strip()

In [ ]:
# Create simple-just annual Compustat variables
# could make it computational easier with unique shift() per variable lagged but did not for readibility of variable creation

funda['bm'] = funda['ceq']/funda['mve_f']
funda['ep'] = funda['ib']/funda['mve_f']
funda['cashpr'] = ((funda['mve_f']+funda['dltt']-funda['at'])/funda['che'])
funda['dy'] = funda['dvt']/funda['mve_f']
funda['lev'] = funda['lt']/funda['mve_f']
funda['sp'] = funda['sale']/funda['mve_f']
funda['roic'] = (funda['ebit']-funda['nopi'])/(funda['ceq']+funda['lt']-funda['che'])
funda['rd_sale'] = funda['xrd']/funda['sale']
funda['rd_mve'] = funda['xrd']/funda['mve_f']
funda['agr'] = funda['at']/funda.groupby('gvkey')['at'].shift(1) - 1
funda['gma'] = (funda['revt']-funda['cogs'])/funda.groupby('gvkey')['at'].shift(1)
funda['chcsho'] = funda['csho']/funda.groupby('gvkey')['csho'].shift(1) - 1 
funda['lgr'] = funda['lt']/funda.groupby('gvkey')['lt'].shift(1)-1
funda['acc'] = np.where(funda['oancf'].notna(), 
                        (funda['ib']-funda['oancf'])/((funda['at']+funda.groupby('gvkey')['at'].shift(1))/2), 
                        ((funda['act']-funda.groupby('gvkey')['act'].shift(1) - (funda['che']-funda.groupby('gvkey')['che'].shift(1))) - ((funda['lct']-funda.groupby('gvkey')['lct'].shift(1))-(funda['dlc']-funda.groupby('gvkey')['dlc'].shift(1))-(funda['txp']-funda.groupby('gvkey')['txp'].shift(1))-funda['dp']))/((funda['at']+funda.groupby('gvkey')['at'].shift(1))/2))
funda['pctacc'] = np.where(funda['oancf'].notna(),
                        np.where(funda['ib']==0, (funda['ib']-funda['oancf'])/0.01, (funda['ib']-funda['oancf'])/funda['ib'].abs()),
                        np.where(funda['ib']==0,
                                ((funda['act']-funda.groupby('gvkey')['act'].shift(1) - (funda['che']-funda.groupby('gvkey')['che'].shift(1))) - ((funda['lct']-funda.groupby('gvkey')['lct'].shift(1))-(funda['dlc']-funda.groupby('gvkey')['dlc'].shift(1))-(funda['txp']-funda.groupby('gvkey')['txp'].shift(1))-funda['dp']))/0.01,
                                ((funda['act']-funda.groupby('gvkey')['act'].shift(1) - (funda['che']-funda.groupby('gvkey')['che'].shift(1))) - ((funda['lct']-funda.groupby('gvkey')['lct'].shift(1))-(funda['dlc']-funda.groupby('gvkey')['dlc'].shift(1))-(funda['txp']-funda.groupby('gvkey')['txp'].shift(1))-funda['dp']))/funda['ib'].abs()))
funda['cfp'] = np.where(funda['oancf'].notna(),
                        funda['oancf']/funda['mve_f'],
                        (funda['ib']-((funda['act']-funda.groupby('gvkey')['act'].shift(1) - (funda['che']-funda.groupby('gvkey')['che'].shift(1))) - ((funda['lct']-funda.groupby('gvkey')['lct'].shift(1))-(funda['dlc']-funda.groupby('gvkey')['dlc'].shift(1))-(funda['txp']-funda.groupby('gvkey')['txp'].shift(1))-funda['dp'])))/funda['mve_f'])
funda['absacc'] = funda['acc'].abs()
funda['age'] = funda['count']
funda['chinv'] = (funda['invt']-funda.groupby('gvkey')['invt'].shift(1))/((funda['at']+funda.groupby('gvkey')['at'].shift(1))/2)
funda['hire'] = np.where(funda['emp'].isna() | funda.groupby('gvkey')['emp'].shift(1).isna(),
                         0,
                         (funda['emp']-funda.groupby('gvkey')['emp'].shift(1))/funda.groupby('gvkey')['emp'].shift(1))
funda['sgr'] = funda['sale']/funda.groupby('gvkey')['sale'].shift(1) - 1
funda['chpm'] = (funda['ib']/funda['sale']) - (funda.groupby('gvkey')['ib'].shift(1)/funda.groupby('gvkey')['sale'].shift(1))
funda['chato'] = (funda['sale']/((funda['at']+funda.groupby('gvkey')['at'].shift(1))/2)) - (funda.groupby('gvkey')['sale'].shift(1)/((funda.groupby('gvkey')['at'].shift(1)+funda.groupby('gvkey')['at'].shift(2))/2))
funda['pchsale_pchinvt'] = ((funda['sale']-funda.groupby('gvkey')['sale'].shift(1))/funda.groupby('gvkey')['sale'].shift(1))-((funda['invt']-funda.groupby('gvkey')['invt'].shift(1))/funda.groupby('gvkey')['invt'].shift(1))
funda['pchsale_pchrect'] = ((funda['sale']-funda.groupby('gvkey')['sale'].shift(1))/funda.groupby('gvkey')['sale'].shift(1))-((funda['rect']-funda.groupby('gvkey')['rect'].shift(1))/funda.groupby('gvkey')['rect'].shift(1))
funda['pchgm_pchsale'] = (((funda['sale']-funda['cogs'])-(funda.groupby('gvkey')['sale'].shift(1)-funda.groupby('gvkey')['cogs'].shift(1)))/(funda.groupby('gvkey')['sale'].shift(1)-funda.groupby('gvkey')['cogs'].shift(1)))-((funda['sale']-funda.groupby('gvkey')['sale'].shift(1))/funda.groupby('gvkey')['sale'].shift(1))
funda['pchsale_pchxsga'] = ((funda['sale']-funda.groupby('gvkey')['sale'].shift(1))/funda.groupby('gvkey')['sale'].shift(1))-((funda['xsga']-funda.groupby('gvkey')['xsga'].shift(1))/funda.groupby('gvkey')['xsga'].shift(1))
funda['depr'] = funda['dp']/funda['ppent']
funda['pchdepr'] = ((funda['dp']/funda['ppent'])-(funda.groupby('gvkey')['dp'].shift(1)/funda.groupby('gvkey')['ppent'].shift(1)))/(funda.groupby('gvkey')['dp'].shift(1)/funda.groupby('gvkey')['ppent'].shift(1))
funda['invest'] = np.where(funda['ppegt'].isnull(),
                           ((funda['ppent']-funda.groupby('gvkey')['ppent'].shift(1))+(funda['invt']-funda.groupby('gvkey')['invt'].shift(1)))/funda.groupby('gvkey')['at'].shift(1),
                           ((funda['ppegt']-funda.groupby('gvkey')['ppegt'].shift(1))+(funda['invt']-funda.groupby('gvkey')['invt'].shift(1)))/funda.groupby('gvkey')['at'].shift(1))
funda['egr'] = (funda['ceq']-funda.groupby('gvkey')['ceq'].shift(1))/funda.groupby('gvkey')['ceq'].shift(1)
funda['capx'] = np.where(funda['capx'].isnull() & (funda['count']>=2),
                         funda['ppent']-funda.groupby('gvkey')['ppent'].shift(1),
                         funda['capx'])
funda['pchcapx'] = (funda['capx']-funda.groupby('gvkey')['capx'].shift(1))/funda.groupby('gvkey')['capx'].shift(1)
funda['grcapx'] = (funda['capx']-funda.groupby('gvkey')['capx'].shift(2))/funda.groupby('gvkey')['capx'].shift(2)
funda['tang'] = (funda['che']+funda['rect']*0.715+funda['invt']*0.547+funda['ppent']*0.535)/funda['at']
funda['sin'] = np.where(funda['sic'].between(2100,2199) | funda['sic'].between(2080,2085) | funda['naics'].isin(['7132','71312','713210','71329','713290','72112','721120']),
                        1,
                        0)
funda['act'] = np.where(funda['act'].isnull(), 
                        funda['che']+funda['rect']+funda['invt'], 
                        funda['act'])
funda['lct'] = np.where(funda['lct'].isnull(), 
                        funda['ap'], 
                        funda['lct'])
funda['currat'] = funda['act']/funda['lct']
funda['pchcurrat'] = ((funda['act']/funda['lct'])-(funda.groupby('gvkey')['act'].shift(1)/funda.groupby('gvkey')['lct'].shift(1)))/(funda.groupby('gvkey')['act'].shift(1)/funda.groupby('gvkey')['lct'].shift(1))
funda['quick'] = (funda['act']-funda['invt'])/funda['lct']
funda['pchquick'] = (((funda['act']-funda['invt'])/funda['lct']) - ((funda.groupby('gvkey')['act'].shift(1)-funda.groupby('gvkey')['invt'].shift(1))/funda.groupby('gvkey')['lct'].shift(1))) / ((funda.groupby('gvkey')['act'].shift(1)-funda.groupby('gvkey')['invt'].shift(1))/funda.groupby('gvkey')['lct'].shift(1))
funda['salecash'] = funda['sale']/funda['che']
funda['salerec'] = funda['sale']/funda['rect']
funda['saleinv'] = funda['sale']/funda['invt']
funda['pchsaleinv'] = (((funda['sale']/funda['invt'])-(funda.groupby('gvkey')['sale'].shift(1)/funda.groupby('gvkey')['invt'].shift(1)))) / (funda.groupby('gvkey')['sale'].shift(1)/funda.groupby('gvkey')['invt'].shift(1))
funda['cashdebt'] = (funda['ib']+funda['dp'])/((funda['lt']+funda.groupby('gvkey')['lt'].shift(1))/2)
funda['realestate'] = np.where(funda['ppegt'].isnull(),
                               (funda['fatb']+funda['fatl'])/funda['ppent'],
                               (funda['fatb']+funda['fatl'])/funda['ppegt'])
funda['divi'] = np.where((funda['dvt'].notna() & (funda['dvt']>0)) & (funda.groupby('gvkey')['dvt'].shift(1).isnull() | (funda.groupby('gvkey')['dvt'].shift(1)==0)), 
                         1, 
                         0)
funda['divo'] = np.where((funda['dvt'].isnull() | (funda['dvt']==0)) & (funda.groupby('gvkey')['dvt'].shift(1).notna() & (funda.groupby('gvkey')['dvt'].shift(1)>0)), 
                         1, 
                         0)
funda['securedind'] = np.where(funda['dm'].notna() & (funda['dm'] != 0), 
                               1, 
                               0)
funda['secured'] = funda['dm']/funda['dltt']
funda['convind'] = np.where((funda['dc'].notna() & (funda['dc']!=0)) | (funda['cshrc'].notna() & (funda['cshrc']!=0)), 
                            1, 
                            0)
funda['grltnoa'] = ((funda['rect']+funda['invt']+funda['ppent']+funda['aco']+funda['intan']+funda['ao']-funda['ap']-funda['lco']-funda['lo'])- (funda.groupby('gvkey')['rect'].shift(1)+funda.groupby('gvkey')['invt'].shift(1)+funda.groupby('gvkey')['ppent'].shift(1)+funda.groupby('gvkey')['aco'].shift(1)+funda.groupby('gvkey')['intan'].shift(1)+funda.groupby('gvkey')['ao'].shift(1)-funda.groupby('gvkey')['ap'].shift(1)-funda.groupby('gvkey')['lco'].shift(1)-funda.groupby('gvkey')['lo'].shift(1))- ((funda['rect']-funda.groupby('gvkey')['rect'].shift(1))+(funda['invt']-funda.groupby('gvkey')['invt'].shift(1))+(funda['aco']-funda.groupby('gvkey')['aco'].shift(1))-((funda['ap']-funda.groupby('gvkey')['ap'].shift(1))+(funda['lco']-funda.groupby('gvkey')['lco'].shift(1)))-funda['dp']))/((funda['at']+funda.groupby('gvkey')['at'].shift(1))/2)

funda['rd'] = np.where((((funda['xrd']/funda['at']) - (funda.groupby('gvkey')['xrd'].shift(1)/funda.groupby('gvkey')['at'].shift(1))) / (funda.groupby('gvkey')['xrd'].shift(1)/funda.groupby('gvkey')['at'].shift(1))) > 0.05, 
                       1, 
                       0) # Prof. Green agreed of this corrected version
funda['operprof'] = (funda['revt']-funda['cogs']-funda['xsga0']-funda['xint0'])/funda.groupby('gvkey')['ceq'].shift(1)
funda['ps'] = ((funda['ni']>0)
              + (funda['oancf']>0)
              + ((funda['ni']/funda['at']) > (funda.groupby('gvkey')['ni'].shift(1)/funda.groupby('gvkey')['at'].shift(1)))
              + (funda['oancf']>funda['ni'])
              + ((funda['dltt']/funda['at']) < (funda.groupby('gvkey')['dltt'].shift(1)/funda.groupby('gvkey')['at'].shift(1)))
              + ((funda['act']/funda['lct']) > (funda.groupby('gvkey')['act'].shift(1)/funda.groupby('gvkey')['lct'].shift(1)))
              + (((funda['sale']-funda['cogs'])/funda['sale']) > ((funda.groupby('gvkey')['sale'].shift(1)-funda.groupby('gvkey')['cogs'].shift(1))/funda.groupby('gvkey')['sale'].shift(1)))
              + ((funda['sale']/funda['at']) > (funda.groupby('gvkey')['sale'].shift(1)/funda.groupby('gvkey')['at'].shift(1)))
              + (funda['scstkc']==0))    

# prep for tb
funda['tr'] = np.select(
    [funda['fyear']<=1978,
     (funda['fyear']>=1979) & (funda['fyear']<=1986),
     funda['fyear']==1987,
     (funda['fyear']>=1988) & (funda['fyear']<=1992),
     funda['fyear']>=1993],
    [0.48, 0.46, 0.40, 0.34, 0.35],
    default=np.nan
)
funda['tb_1'] = np.where(funda['txfo'].isnull() | funda['txfed'].isnull(),
                          ((funda['txt']-funda['txdi'])/funda['tr'])/funda['ib'],
                          ((funda['txfo']+funda['txfed'])/funda['tr'])/funda['ib'])
funda['tb_1'] = np.where((((funda['txfo']+funda['txfed'])>0) | (funda['txt']>funda['txdi'])) & (funda['ib']<=0),
                          1,
                          funda['tb_1'])

# prep for ms
funda['roa'] = funda['ni']/((funda['at']+funda.groupby('gvkey')['at'].shift(1))/2)
funda['cfroa'] = np.where(funda['oancf'].isnull(),
                          (funda['ib']+funda['dp'])/((funda['at']+funda.groupby('gvkey')['at'].shift(1))/2),
                          funda['oancf']/((funda['at']+funda.groupby('gvkey')['at'].shift(1))/2))
funda['xrdint'] = funda['xrd']/((funda['at']+funda.groupby('gvkey')['at'].shift(1))/2)
funda['capxint'] = funda['capx']/((funda['at']+funda.groupby('gvkey')['at'].shift(1))/2)
funda['xadint'] = funda['xad']/((funda['at']+funda.groupby('gvkey')['at'].shift(1))/2)
